# 02 — Experiment and train models using features

Adapted from the official [2.Experiment-train-models-using-features.ipynb](https://github.com/Azure/azureml-examples/blob/65eceac7ff5cd2417af5f24d38192ab8e445dd85/sdk/python/featurestore_sample/notebooks/sdk_only/2.Experiment-train-models-using-features.ipynb) notebook
(MIT). The official code is kept as-is except for the local-execution changes listed in the
[README](README.md#changes-from-the-official-notebooks). Talk track: Slides 15–18 in [FEATURE_STORE_TALK_TRACK.md](FEATURE_STORE_TALK_TRACK.md).

Experiment with unregistered features using **local Spark**, register the `accounts` feature set, export a feature retrieval specification, and train with a pipeline whose built-in feature retrieval component reads the **materialized offline store on serverless Spark**.

| Label | Where the work runs |
|---|---|
| **[Local]** | Your computer: the notebook kernel, local Spark, or local Python |
| **[Control plane]** | Azure ML or Azure Resource Manager API call made from your computer |
| **[Serverless Spark]** | Azure ML managed Spark job submitted from your computer |

In this tutorial series you will experience how features seamlessly integrates all the phases of ML lifecycle: Prototyping features, training and operationalizing.

In part 1 of the tutorial you learnt how to create a feature set spec with custom transformations, enable materialization and perform backfill. In this tutorial you will will learn how to experiment with features to improve model performance. You will see how feature store increasing agility in the experimentation and training flows. 

You will perform the following:
- Prototype a create new `acccounts` feature set spec using existing precomputed values as features, unlike part 1 of the tutorial where we created feature set that had custom transformations. You will then Register the local feature set spec as a feature set in the feature store
- Select features for the model: You will select features from the `transactions` and `accounts` feature sets and save them as a feature-retrieval spec
- Run training pipeline that uses the Feature retrieval spec to train a new model. This pipeline will use the built in feature-retrieval component to generate the training data

## Prerequisites

- Run the notebooks in order from this folder with the repository `.venv` kernel. Run
  [00_prepare_sample_data.ipynb](00_prepare_sample_data.ipynb) first.
- Sign in with `az login`. The notebooks use `AzureCliCredential` instead of `AzureMLOnBehalfOfCredential`,
  which works only inside Azure ML compute.
- Configure the feature store section of `workshop/.env`; see [README](README.md#configuration).
- Your account needs `Owner`, or `Contributor` plus `User Access Administrator`, on the resource group.

#### **[Local]** Start a local Spark session
The official notebook attaches **Serverless Spark Compute** in Azure ML studio, which provides `spark` and the `AZUREML_ARM_*` environment variables. This cell provides the same values locally from `workshop/.env` and starts Spark on this computer.

In [1]:
# Local equivalent of the Azure ML studio Serverless Spark notebook session.
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

load_dotenv(Path.cwd().parents[1] / ".env", override=True)
os.environ["AZUREML_ARM_SUBSCRIPTION"] = os.environ["AZURE_SUBSCRIPTION_ID"]
os.environ["AZUREML_ARM_RESOURCEGROUP"] = os.environ["AZURE_RESOURCE_GROUP"]
os.environ["AZUREML_ARM_WORKSPACE_NAME"] = os.environ["AZUREML_WORKSPACE_NAME"]

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PATH"] = str(Path(os.environ["HADOOP_HOME"]) / "bin") + os.pathsep + os.environ["PATH"]
spark = (
    SparkSession.builder.appName("featurestore-local")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)

# Local copy for local Spark development, and the ADLS Gen2 copy used by registered feature sets.
local_data_root = (Path.cwd().parents[1] / "outputs" / "feature_store_sample_data" / "feature-store-prp").as_uri()
cloud_data_root = (
    f"abfss://{os.environ['FEATURE_STORE_SOURCE_FILESYSTEM']}"
    f"@{os.environ['FEATURE_STORE_SOURCE_STORAGE_ACCOUNT']}.dfs.core.windows.net/feature-store-prp"
)
print("start spark session" if "spark" in globals() else "local configuration loaded")

start spark session


#### Setup root directory for the samples

In [2]:
import os

# Local folder with the official featurestore_sample assets (featurestore/ and project/).
root_dir = os.path.join(os.getcwd(), "featurestore_sample")

if os.path.isdir(root_dir):
    print("The folder exists.")
else:
    print("The folder does not exist. Please create or fix the path")

The folder exists.


#### Initialize the project workspace CRUD client
This is the current workspace where you will be running the tutorial notebook from

In [3]:
### Initialize the MLClient of this project workspace
import os
from azure.ai.ml import MLClient
from azure.identity import AzureCliCredential

project_ws_sub_id = os.environ["AZUREML_ARM_SUBSCRIPTION"]
project_ws_rg = os.environ["AZUREML_ARM_RESOURCEGROUP"]
project_ws_name = os.environ["AZUREML_ARM_WORKSPACE_NAME"]

# connect to the project workspace
ws_client = MLClient(
    AzureCliCredential(process_timeout=60), project_ws_sub_id, project_ws_rg, project_ws_name
)

#### Initialize the feature store CRUD client
Ensure you update the `featurestore_name` to reflect what you created in part 1 of this tutorial

In [4]:
from azure.ai.ml import MLClient
from azure.identity import AzureCliCredential

# feature store
featurestore_name = (
    os.environ["AZUREML_FEATURE_STORE_NAME"]  # use the same name from part #1 of the tutorial
)
featurestore_subscription_id = os.environ["AZUREML_ARM_SUBSCRIPTION"]
featurestore_resource_group_name = os.environ["AZUREML_ARM_RESOURCEGROUP"]

# feature store ml client
fs_client = MLClient(
    AzureCliCredential(process_timeout=60),
    featurestore_subscription_id,
    featurestore_resource_group_name,
    featurestore_name,
)

Overriding of current TracerProvider is not allowed


Overriding of current LoggerProvider is not allowed


Overriding of current MeterProvider is not allowed


Attempting to instrument while already instrumented


Attempting to instrument while already instrumented


Attempting to instrument while already instrumented


Attempting to instrument while already instrumented


Attempting to instrument while already instrumented


#### Initialize the feature store core sdk client

In [5]:
# feature store client
from azureml.featurestore import FeatureStoreClient
from azure.identity import AzureCliCredential

featurestore = FeatureStoreClient(
    credential=AzureCliCredential(process_timeout=60),
    subscription_id=featurestore_subscription_id,
    resource_group_name=featurestore_resource_group_name,
    name=featurestore_name,
)

Overriding of current TracerProvider is not allowed


Overriding of current LoggerProvider is not allowed


Overriding of current MeterProvider is not allowed


Attempting to instrument while already instrumented


Attempting to instrument while already instrumented


Attempting to instrument while already instrumented


Attempting to instrument while already instrumented


Attempting to instrument while already instrumented


#### Create compute cluster with name `cpu-cluster-fs` in the project workspace
This will be needed when we run training/batch inference jobs

In [6]:
from azure.ai.ml.entities import AmlCompute

# This workspace's container registry has no admin user, so the cluster pulls images with the workspace
# compute identity, which has AcrPull. Jobs also use this identity to read credential-less datastores.
compute_identity = ws_client.compute.get(os.environ["AZUREML_COMPUTE_NAME"]).identity

cluster_basic = AmlCompute(
    name="cpu-cluster-fs",
    type="amlcompute",
    size="STANDARD_F4S_V2",  # you can replace it with other supported VM SKUs
    location=ws_client.workspaces.get(ws_client.workspace_name).location,
    min_instances=0,
    max_instances=1,
    idle_time_before_scale_down=360,
    identity=compute_identity,
    tags={"SecurityControl": "Ignore"},
)
ws_client.begin_create_or_update(cluster_basic).result()

AmlCompute({'type': 'amlcompute', 'created_on': None, 'provisioning_state': 'Succeeded', 'provisioning_errors': None, 'name': 'cpu-cluster-fs', 'description': None, 'tags': {'SecurityControl': 'Ignore'}, 'properties': {}, 'print_as_yaml': False, 'id': '/subscriptions/{subscription-id}/resourceGroups/rg-aml-ws-public-ncus-04/providers/Microsoft.MachineLearningServices/workspaces/mlwpublicncus04/computes/cpu-cluster-fs', 'Resource__source_path': '', 'base_path': 'C:\\Users\\{user}\\Documents\\MLOPs-AzureML\\workshop\\notebooks\\07_feature_store', 'creation_context': None, 'serialize': <msrest.serialization.Serializer object at 0x000002AD7D181220>, 'resource_id': None, 'location': 'northcentralus', 'size': 'STANDARD_F4S_V2', 'min_instances': 0, 'max_instances': 1, 'idle_time_before_scale_down': 360.0, 'identity': <azure.ai.ml.entities._credentials.IdentityConfiguration object at 0x000002AD6B84BEF0>, 'ssh_public_access_enabled': True, 'ssh_settings': None, 'network_settings': <azure.ai.ml.

## Step 1: Create accounts featureset locally from precomputed data
In tutorial part 1, we created a transactions featureset that had custom transformations. Now we will create an accounts featureset that will use precomputed values. 

For onboarding precomputed features, you can create a featureset spec without writing any transformation code. Featureset spec is a specification to develop and test a featureset in a fully local/dev environment without connecting to any featurestore. In this step you will create the feature set spec locally and sample the values from it. If you want to get managed featurestore capabilities, you need to register the featureset spec with a feature store using a feature asset definition (a future step in this tutorial).

#### Step 1a: **[Local]** Explore the source data for accounts

##### Note
Local Spark reads the local copy of the sample data from notebook 00. Registered feature sets use the ADLS Gen2
copy.

In [7]:
accounts_data_path = local_data_root + "/datasources/accounts-precalculated/*.parquet"
accounts_df = spark.read.parquet(accounts_data_path)

display(accounts_df.head(5))

[Row(accountID='A985156952816816', timestamp=datetime.datetime(2021, 12, 31, 19, 0), accountCountry='US', isUserRegistered=True, numPaymentRejects1dPerUser=0.0, accountAge=312.0),
 Row(accountID='A1055521248929430', timestamp=datetime.datetime(2021, 12, 31, 19, 0), accountCountry='BR', isUserRegistered=False, numPaymentRejects1dPerUser=0.0, accountAge=1337.0),
 Row(accountID='A914800935560176', timestamp=datetime.datetime(2021, 12, 31, 19, 0), accountCountry='US', isUserRegistered=False, numPaymentRejects1dPerUser=3.0, accountAge=41.0),
 Row(accountID='A1055521510328860', timestamp=datetime.datetime(2021, 12, 31, 19, 0), accountCountry='CA', isUserRegistered=False, numPaymentRejects1dPerUser=1.0, accountAge=612.0),
 Row(accountID='A1008037957659561', timestamp=datetime.datetime(2021, 12, 31, 19, 0), accountCountry='BR', isUserRegistered=True, numPaymentRejects1dPerUser=1.0, accountAge=894.0)]

#### Step 1b: Create `accounts` feature set spec in local from these precomputed features
Note that we do not need any transformation code here since we are referencing precomputed features.

In [8]:
from azureml.featurestore import create_feature_set_spec, FeatureSetSpec
from azureml.featurestore.contracts import (
    DateTimeOffset,
    Column,
    ColumnType,
    SourceType,
    TimestampColumn,
)
from azureml.featurestore.feature_source import ParquetFeatureSource


accounts_featureset_spec = create_feature_set_spec(
    source=ParquetFeatureSource(
        path=accounts_data_path,
        timestamp_column=TimestampColumn(name="timestamp"),
    ),
    index_columns=[Column(name="accountID", type=ColumnType.string)],
    # account profiles in the source are updated once a year. set temporal_join_lookback to 365 days
    temporal_join_lookback=DateTimeOffset(days=365, hours=0, minutes=0),
    infer_schema=True,
)

In [9]:
# Generate a spark dataframe from the feature set specification
accounts_fset_df = accounts_featureset_spec.to_spark_dataframe()
# display few records
display(accounts_fset_df.head(5))

[Row(accountID='A985156952816816', timestamp=datetime.datetime(2021, 12, 31, 19, 0), accountCountry='US', isUserRegistered=True, numPaymentRejects1dPerUser=0.0, accountAge=312.0),
 Row(accountID='A1055521248929430', timestamp=datetime.datetime(2021, 12, 31, 19, 0), accountCountry='BR', isUserRegistered=False, numPaymentRejects1dPerUser=0.0, accountAge=1337.0),
 Row(accountID='A914800935560176', timestamp=datetime.datetime(2021, 12, 31, 19, 0), accountCountry='US', isUserRegistered=False, numPaymentRejects1dPerUser=3.0, accountAge=41.0),
 Row(accountID='A1055521510328860', timestamp=datetime.datetime(2021, 12, 31, 19, 0), accountCountry='CA', isUserRegistered=False, numPaymentRejects1dPerUser=1.0, accountAge=612.0),
 Row(accountID='A1008037957659561', timestamp=datetime.datetime(2021, 12, 31, 19, 0), accountCountry='BR', isUserRegistered=True, numPaymentRejects1dPerUser=1.0, accountAge=894.0)]

#### Step 1c:  Export as feature set spec
In order to register the feature set spec with the feature store, it needs to be saved in a specific format. 
Action: After running the below cell, please inspect the generated `accounts` FeatureSetSpec: Open this file from the file tree to see the spec: `featurestore/featuresets/accounts/spec/FeatureSetSpec.yaml`

Spec contains these important elements:

1. `source`: reference to a storage. In this case a parquet file in a blob storage.
1. `features`: list of features and their datatypes. If you provide transformation code (see Day 2 section), the code has to return a dataframe that maps to the features and datatypes. In case where you do not provide transformation code (in this case of `accounts` because it is precomputed), the system will build the query to to map these to the source 
1. `index_columns`: the join keys required to access values from the feature set

Learn more about it in the [top level feature store entities document](fs-concepts-todo) and the [feature set spec yaml reference](reference-yaml-featureset-spec.md).

The additional benefit of persisting it is that it can be source controlled.

In [10]:
# Export the specification with the ADLS Gen2 copy of the source data, as in notebook 01.
accounts_featureset_spec.source.path = cloud_data_root + "/datasources/accounts-precalculated/*.parquet"
print(accounts_featureset_spec.source.path)

abfss://spark@stpublicdlncus04.dfs.core.windows.net/feature-store-prp/datasources/accounts-precalculated/*.parquet


In [11]:
import os

# create a new folder to dump the feature set spec
accounts_featureset_spec_folder = root_dir + "/featurestore/featuresets/accounts/spec"

# check if the folder exists, create one if not
if not os.path.exists(accounts_featureset_spec_folder):
    os.makedirs(accounts_featureset_spec_folder)

accounts_featureset_spec.dump(accounts_featureset_spec_folder, overwrite=True)

In [12]:
# Keep experimenting with local Spark: point the in-memory specification back to the local copy.
accounts_featureset_spec.source.path = accounts_data_path

## Step 2: **[Local]** Experiment with unregistered features locally and register with feature store when ready

The official notebook combines the local `accounts` specification with the registered `transactions` feature set.
Local Spark can't read the registered, identity-protected data, so this notebook loads the `transactions`
specification exported by notebook 01 and points it at the local copy. The features and point-in-time join are
the same.

#### Step 2a: Select features for model

In [13]:
# Load the transactions specification exported by notebook 01 and use the local copy of its source data.
transactions_featureset_spec = FeatureSetSpec.from_config(root_dir + "/featurestore/featuresets/transactions/spec")
transactions_featureset_spec.source.path = local_data_root + "/datasources/transactions-source/*.parquet"
# Notice that account feature set spec is in your local dev environment (this notebook): not registered with feature store yet
features = [
    accounts_featureset_spec.get_feature("accountAge"),
    accounts_featureset_spec.get_feature("numPaymentRejects1dPerUser"),
    transactions_featureset_spec.get_feature("transaction_amount_7d_sum"),
    transactions_featureset_spec.get_feature("transaction_amount_3d_sum"),
    transactions_featureset_spec.get_feature("transaction_amount_7d_avg"),
]

#### Step 2b: Generate training data locally
In this step we generate training data for illustrative purpose. You can optionally train models locally with this. In the upcoming steps in this tutorial, you will train a model in the cloud.

In [14]:
from azureml.featurestore import get_offline_features

# Load the observation data. To understand observatio ndata, refer to part 1 of this tutorial
observation_data_path = local_data_root + "/observation_data/train/*.parquet"
observation_data_df = spark.read.parquet(observation_data_path)
obs_data_timestamp_column = "timestamp"

In [15]:
# generate training dataframe by using feature data and observation data
training_df = get_offline_features(
    features=features,
    observation_data=observation_data_df,
    timestamp_column=obs_data_timestamp_column,
)

# Ignore the message that says feature set is not materialized (materialization is optional). We will enable materialization in the next part of the tutorial.
display(training_df)
# Note: display(training_df.head(5)) displays the timestamp column in a different format. You can can call training_df.show() to see correctly formatted value

DataFrame[transactionID: string, accountID: string, timestamp: timestamp, transactionAmount: double, localHour: int, transactionAmountUSD: double, digitalItemCount: bigint, physicalItemCount: bigint, isProxyIP: boolean, is_fraud: boolean, accountAge: double, numPaymentRejects1dPerUser: double, transaction_amount_7d_sum: double, transaction_amount_3d_sum: double, transaction_amount_7d_avg: double]

In [16]:
training_df.show(5)

+-------------+-----------------+-------------------+-----------------+---------+--------------------+----------------+-----------------+---------+--------+----------+--------------------------+-------------------------+-------------------------+-------------------------+
|transactionID|        accountID|          timestamp|transactionAmount|localHour|transactionAmountUSD|digitalItemCount|physicalItemCount|isProxyIP|is_fraud|accountAge|numPaymentRejects1dPerUser|transaction_amount_7d_sum|transaction_amount_3d_sum|transaction_amount_7d_avg|
+-------------+-----------------+-------------------+-----------------+---------+--------------------+----------------+-----------------+---------+--------+----------+--------------------------+-------------------------+-------------------------+-------------------------+
|   T000381451|A6310357522954187|2023-01-08 00:16:45|            43.64|        0|               42.94|               2|                1|    false|   false|     380.0|              

#### Step 2c: Register the `accounts` featureset with the featurestore
Once you have experimented with different feature definitions locally and sanity tested it, you can register it with the feature store.
For this you will register a featureset asset definition with the feature store.


In [17]:
from azure.ai.ml.entities import FeatureSet, FeatureSetSpecification

accounts_fset_config = FeatureSet(
    name="accounts",
    version="1",
    description="accounts featureset",
    entities=[f"azureml:account:1"],
    stage="Development",
    specification=FeatureSetSpecification(path=accounts_featureset_spec_folder),
    tags={"data_type": "nonPII"},
)

poller = fs_client.feature_sets.begin_create_or_update(accounts_fset_config)
print(poller.result())

Uploading spec (0.0 MBs):   0%|          | 0/508 [00:00<?, ?it/s]

Uploading spec (0.0 MBs): 100%|##########| 508/508 [00:00<00:00, 6308.72it/s]




Readonly attribute name will be ignored in class <class 'azure.ai.ml._restclient.v2023_10_01.models._models_py3.FeaturesetVersion'>


description: accounts featureset
entities:
- azureml:account:1
name: accounts
specification:
  path: azureml://subscriptions/{subscription-id}/resourcegroups/rg-aml-ws-public-ncus-04/workspaces/mlfspublicncus04/datastores/workspaceblobstore/paths/LocalUpload/d96f802955021a95303af1bb9d9411565a36496a144a2b4919053c43a6b370e7/spec
stage: Development
tags:
  data_type: nonPII
version: '1'



#### Step 2d: Get registered featureset and sanity test

In [18]:
# look up the featureset by providing name and version
accounts_featureset = featurestore.feature_sets.get("accounts", "1")

In [19]:
# The registered feature set's data is read on serverless Spark in Step 4. Locally, list its features.
accounts_featureset.features

[Feature(Name=accountCountry,Type=string),
 Feature(Name=isUserRegistered,Type=boolean),
 Feature(Name=numPaymentRejects1dPerUser,Type=double),
 Feature(Name=accountAge,Type=double)]

## Step 3: Run training experiment
In this step you will select a list of features, run a training pipeline, and register the model. You can repeat this step till you are happy with the model performance.

#### (Optional) Step 3a: Discover features from Feature Store UI
You have already done this in part 1 of the tutorial after registering the `transactions` feature set. Since you also have `accounts` featureset, you can browse the available features:
* Goto the [Azure ML global landing page](https://ml.azure.com/home?flight=FeatureStores).
* Click on `Feature stores` in the left nav
* You will see the list of feature stores that you have access to. Click on the feature store that you created above.

You can see the feature sets and entity that you created. Click on the feature sets to browse the feature definitions. You can also search for feature  sets across feature stores by using the global search box.

#### (Optional) Step 3b: Discover features from SDK

In [20]:
# List available feature sets
all_featuresets = featurestore.feature_sets.list()
for fs in all_featuresets:
    print(fs)

# List of versions for transactions feature set
all_transactions_featureset_versions = featurestore.feature_sets.list(
    name="transactions"
)
for fs in all_transactions_featureset_versions:
    print(fs)

# See properties of the transactions featureset including list of features
featurestore.feature_sets.get(name="transactions", version="1").features

latest_version: '1'
name: accounts
stage: Development

latest_version: '1'
name: transactions
stage: Development



{'additional_properties': {}, 'id': '/subscriptions/{subscription-id}/resourceGroups/rg-aml-ws-public-ncus-04/providers/Microsoft.MachineLearningServices/workspaces/mlfspublicncus04/featuresets/transactions/versions/1', 'name': '1', 'type': 'Microsoft.MachineLearningServices/workspaces/featuresets/versions', 'system_data': <azure.ai.ml._restclient.v2023_10_01.models._models_py3.SystemData object at 0x0000025AB373F560>, 'properties': <azure.ai.ml._restclient.v2023_10_01.models._models_py3.FeaturesetVersionProperties object at 0x0000025AB3961FA0>}


[Feature(Name=transaction_3d_count,Type=long),
 Feature(Name=transaction_amount_3d_sum,Type=double),
 Feature(Name=transaction_amount_3d_avg,Type=double),
 Feature(Name=transaction_7d_count,Type=long),
 Feature(Name=transaction_amount_7d_sum,Type=double),
 Feature(Name=transaction_amount_7d_avg,Type=double)]

#### Step 3c: Select features for the model and export it as a feature-retrieval spec
In the previous steps, you selected features from a combination unregistered  and registered feature sets for local experimentation and testing. Now you are ready to experiment in the cloud. Saving the selected features as a feature-retrieval spec and using it in the mlops/cicd flow for training/inference increases your agility in shipping models.

Select features for the model

In [21]:
# get the registered transactions feature set, version 1
transactions_featureset = featurestore.feature_sets.get("transactions", "1")

# you can select features in pythonic way
features = [
    accounts_featureset.get_feature("accountAge"),
    transactions_featureset.get_feature("transaction_amount_7d_sum"),
    transactions_featureset.get_feature("transaction_amount_3d_sum"),
]

# you can also specify features in string form: featurestore:featureset:version:feature
more_features = [
    f"accounts:1:numPaymentRejects1dPerUser",
    f"transactions:1:transaction_amount_7d_avg",
]

more_features = featurestore.resolve_feature_uri(more_features)

features.extend(more_features)

Export selected features as a feature-retrieval spec

#### Note
Feature retrieval spec is a portable definition of list of features associated with a model. This can help streamline ML model development and operationalizing.This will be an input to the training pipeline (used to generate the training data), then will be packaged along with the model, and will be used during inference to lookup the features. It will be a glue that integrates all phases of the ML lifecycle. Changes to training/inference pipeline can be kept minimal as you experiment and deploy. 

Using feature retrieval spec and the built-in feature retrieval component is optional: you can directly use `get_offline_features()` api as shown above.

Note that the name of the spec should be `feature_retrieval_spec.yaml` when it is packaged with the model for the system to recognize it.

In [22]:
# Create feature retrieval spec
feature_retrieval_spec_folder = root_dir + "/project/fraud_model/feature_retrieval_spec"

# check if the folder exists, create one if not
if not os.path.exists(feature_retrieval_spec_folder):
    os.makedirs(feature_retrieval_spec_folder)

featurestore.generate_feature_retrieval_spec(feature_retrieval_spec_folder, features)

Overriding of current TracerProvider is not allowed


Overriding of current LoggerProvider is not allowed


Overriding of current MeterProvider is not allowed


Attempting to instrument while already instrumented


Attempting to instrument while already instrumented


Attempting to instrument while already instrumented


Attempting to instrument while already instrumented


Attempting to instrument while already instrumented


## Step 4: **[Serverless Spark]** Train in the cloud using pipelines and register model if satisfactory
In this step you will manually trigger the training pipeline. In a production scenario, this could be triggered by a ci/cd pipeline based on changes to the feature-retrieval spec in the source repository.

#### Step 4a: Run the training pipeline
The training pipeline has the following steps:

1. Feature retrieval step: This is a built-in component takes as input the feature retrieval spec, the observation data and timestamp column name. It then generates the training data as output. It runs this as a managed spark job.
1. Training step: This step trains the model based on the training data and generates a model (not registered yet)
1. Evaluation step: This step validates whether model performance/quailty is within threshold (in our case it is a placeholder/dummy step for illustration purpose)
1. Register model step: This step registers the model

Note: In part 2 of this tutorial you ran a backfill job to materialize data for `transactions` feature set. Feature retrieval step will read feature values from offline store for this feature set. The behavior will same even if you use `get_offline_features()` api.

#### **[Control plane]** Register an ADLS Gen2 datastore for Spark pipeline outputs
The built-in feature retrieval step writes its output with Spark through the ADLS Gen2 (`dfs`) endpoint. This
project workspace's default storage account has blob soft delete enabled, which that endpoint rejects
(`This endpoint does not support BlobStorageEvents or SoftDelete`). The pipelines therefore write the retrieval
output to this credential-less datastore on the Spark ADLS Gen2 account. The Spark step uses your identity; the
training and scoring steps on `cpu-cluster-fs` use the compute identity, which gets **Storage Blob Data Reader**.

In [23]:
import time
from uuid import uuid4

from azure.ai.ml.entities import AzureDataLakeGen2Datastore
from azure.core.exceptions import ResourceExistsError
from azure.mgmt.authorization import AuthorizationManagementClient
from azure.mgmt.authorization.models import RoleAssignmentCreateParameters

ws_client.datastores.create_or_update(
    AzureDataLakeGen2Datastore(
        name="featurestore_adls",
        account_name=os.environ["FEATURE_STORE_SOURCE_STORAGE_ACCOUNT"],
        filesystem=os.environ["FEATURE_STORE_SOURCE_FILESYSTEM"],
        description="ADLS Gen2 outputs for feature store Spark pipeline steps",
    )
)

# Command steps on cpu-cluster-fs read the retrieval output with the compute identity.
compute_principal_id = ws_client.compute.get("cpu-cluster-fs").identity.user_assigned_identities[0].principal_id
scope = f"/subscriptions/{project_ws_sub_id}/resourceGroups/{project_ws_rg}/providers/Microsoft.Storage/storageAccounts/{os.environ['FEATURE_STORE_SOURCE_STORAGE_ACCOUNT']}"
# The role definition ID for the "Storage Blob Data Reader" role.
role_definition_id = f"/subscriptions/{project_ws_sub_id}/providers/Microsoft.Authorization/roleDefinitions/2a2b9908-6ea1-4ae2-8e65-a410df84e7d1"
try:
    AuthorizationManagementClient(AzureCliCredential(process_timeout=60), project_ws_sub_id).role_assignments.create(
        scope,
        str(uuid4()),
        RoleAssignmentCreateParameters(
            principal_id=compute_principal_id,
            role_definition_id=role_definition_id,
            principal_type="ServicePrincipal",
        ),
    )
    print("Storage Blob Data Reader granted to the compute identity. Waiting two minutes for the role to propagate.")
    time.sleep(120)
except ResourceExistsError:
    print("Storage Blob Data Reader is already assigned to the compute identity.")

Storage Blob Data Reader granted to the compute identity. Waiting two minutes for the role to propagate.


In [24]:
from azure.ai.ml import load_job  # will be used later

training_pipeline_path = (
    root_dir + "/project/fraud_model/pipelines/training_pipeline.yaml"
)
training_pipeline_definition = load_job(source=training_pipeline_path)
training_pipeline_job = ws_client.jobs.create_or_update(training_pipeline_definition)
ws_client.jobs.stream(training_pipeline_job.name)
# Note: First time it runs, each step in pipeline can take ~ 15 mins. However subsequent runs can be faster (assuming spark pool is warm - default timeout is 30 mins)

Class AutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


Class AutoDeleteConditionSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


Class BaseAutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


Class IntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


Class ProtectionLevelSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


...


Streaming logs/azureml/executionlogs.txt

[2026-10-07 03:42:54Z] Submitting 1 runs, first five are: 3266323f:b8d089bf-1f42-425b-980b-53251c910cc0


[2026-10-07 03:50:15Z] Completing processing run id b8d089bf-1f42-425b-980b-53251c910cc0.
[2026-10-07 03:50:18Z] Submitting 1 runs, first five are: 7d23425a:7cb833c2-6118-491f-80de-fe164c42754d


[2026-10-07 03:54:24Z] Completing processing run id 7cb833c2-6118-491f-80de-fe164c42754d.
[2026-10-07 03:54:30Z] Submitting 1 runs, first five are: c46577aa:58c8a616-525d-4e35-9685-9f84c4c3fb5b


[2026-10-07 03:56:55Z] Completing processing run id 58c8a616-525d-4e35-9685-9f84c4c3fb5b.
[2026-10-07 03:56:58Z] Submitting 1 runs, first five are: b409d080:f9195545-86c5-480b-be40-3c61941cad51


[2026-10-07 03:59:20Z] Completing processing run id f9195545-86c5-480b-be40-3c61941cad51.

Execution Summary
RunId: patient_dog_gm8h3zcgrh
Web View: https://ml.azure.com/runs/patient_dog_gm8h3zcgrh?wsid=/subscriptions/{subscription-id}/resourcegroups/rg-aml-ws-public-ncus-04/workspaces/mlwpublicncus04



#### Inspect the training pipeline and the model
Open the above pipeline run "web view" in new window to see the steps in the pipeline.


#### Step 4b: Notice the feature retrieval spec in the model artifacts
1. In the left nav of the current workspace -> right click on `Models` -> Open in new tab or window
1. Click on `fraud_model`
1. Click on `Artifacts` in the top nav

You can notice that the feature retrieval spec is packaged along with the model. The model registration step in the training pipeline has done this. You created feature retrieval spec during experimentation, now it has become part of the model definition. In the next tutorial you will see how this will be used during inferencing.


## Step 5: View the feature set and model dependencies

#### Step 5a: View the list of feature sets associated with the model
In the same models page, click on the `feature sets` tab. Here you can see both `transactions` and `accounts` featuresets that this model depends on.

#### Step 5b: View the list of models using the feature sets
1. Open the feature store UI (expalined in a previous step in this tutorial)
1. Click on `Feature sets` on the left nav
1. Click on any of the feature set -> click on `Models` tab

You can see the list of models that are using the feature sets (determined from the feature retrieval spec when the model was registered).

## Cleanup

Keep the resources for the next notebooks. See [README](README.md#cleanup) when you finish the demo.

## Next steps
* [03 — Enable recurrent materialization and run batch inference](03_recurrent_materialization_batch_inference.ipynb)